# 02 — EDA and preprocessing

EDA of the IBM transactions and the preprocessing pipeline (`src/preprocessing.py`).

**Key facts:** no natural missing values; amount skew ~219 (log-transform); Account IDs near-unique (dropped); temporal split by day (1-3 reference, 4-10 incoming); use the FULL file, not a head-sample.

In [ ]:
import sys, os
sys.path.append(os.path.abspath('../src'))
import numpy as np, pandas as pd
from preprocessing import load_transactions, temporal_split, build_module1_features
from injection import inject_anomalies
DATA_PATH = '../data/HI-Small_Trans.csv'   # <-- set to your HPC path

In [ ]:
df = load_transactions(DATA_PATH)
print('shape:', df.shape)
print('skew:', round(df['Amount Paid'].skew(),1))
print('natural missing:', int(df.isna().sum().sum()))
print('laundering rate: %.4f%%' % (100*df['Is Laundering'].mean()))

In [ ]:
import matplotlib.pyplot as plt
fig,ax=plt.subplots(1,2,figsize=(11,3.5))
df['Amount Paid'].clip(upper=df['Amount Paid'].quantile(0.99)).hist(bins=60,ax=ax[0]); ax[0].set_title('Amount Paid (raw)')
np.log1p(df['Amount Paid']).hist(bins=60,ax=ax[1]); ax[1].set_title('log1p(Amount Paid)')
plt.tight_layout(); plt.savefig('../results/amount_distribution.png',dpi=120); plt.show()

## Temporal split + features (fit on reference, apply to incoming)

In [ ]:
ref, inc = temporal_split(df, reference_days=3, max_days=10)
print(f'reference: {ref.shape[0]:,}  {ref.Timestamp.min()} -> {ref.Timestamp.max()}')
print(f'incoming : {inc.shape[0]:,}  {inc.Timestamp.min()} -> {inc.Timestamp.max()}')
Xref, scaler, names = build_module1_features(ref, fit=True)
print('features:', names, '| std ~1:', np.round(Xref.std(0),2))

In [ ]:
inc_c, gt = inject_anomalies(inc, 'Amount Paid', rate=0.02, multiplier=50, seed=42)
Xinc,_,_ = build_module1_features(inc_c, scaler=scaler, fit=False)
la=Xinc[:,0]; a=gt.values
print(f'scaled log_amount clean={la[~a].mean():.2f} anomaly={la[a].mean():.2f} sep={la[a].mean()-la[~a].mean():.2f}')